In [17]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm

In [26]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            # 5x5 conv layer
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=0, stride=1),

            # tanh activation 
            nn.Tanh(),

            # avg pool 1
            nn.AvgPool2d(kernel_size=2, stride=2),

            #   5x5 conv layer 
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=0, stride=1),

            # tanh activation 
            nn.Tanh(),

            # avg pool 2
            nn.AvgPool2d(kernel_size=2, stride=2),

            nn.Flatten(),

            # fully connected layer 1
            nn.Linear(in_features=400, out_features=120),

            # tanh activation 
            nn.Tanh(),

            # fully connected layer 2
            nn.Linear(in_features=120, out_features=84),

            # tanh activation 
            nn.Tanh(),

            # fully connected layer 3
            nn.Linear(in_features=84, out_features=10)
        )
        
    def forward(self, x):
        x = self.layers(x)
        return x


In [27]:
class RGBImageDataset(Dataset):
    def __init__(self, num=1000, height=32, width=32):
        self.num = num
        
        self.images = torch.rand(num, 1, height, width)
        
        self.labels = torch.randint(0, 10, (num,))

    def __len__(self):
        return self.num

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [28]:
train_dataset = RGBImageDataset()
val_dataset = RGBImageDataset()
batch_size = 32
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,      
    num_workers=2     
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,      
    num_workers=2     
)


In [29]:
model = LeNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

In [30]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model = model.to(device)
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc=" Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # delete previous grad
        optimizer.zero_grad()
        outputs = model(images)

        # calc loss
        loss = criterion(outputs, labels)

        # backward to calc loss
        loss.backward()

        # optimize model
        optimizer.step()

        # result
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total

In [31]:
def validate(model, loader, criterion, device):

    model = model.to(device)
    # eval mode
    model.eval()

    # init
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=" Validating", leave=False):

            #load to device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total , 100 * correct / total 

    

In [33]:
EPOCHS = 10
best_val_acc = 0.0
device = torch.device("cpu") 
if torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple Silicon GPU


for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)

    print(f"  -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    val_loss, val_acc = validate(model, val_dataloader, criterion, device)
    print(f"  -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        print(f"  [Save] Best val acc: {best_val_acc:.2f}%")
        

  -> Train Loss: 2.3004 | Train Acc: 10.30%


  -> Val Loss: 2.3049 | Val Acc: 9.90%
  [Save] Best val acc: 9.90%


  -> Train Loss: 2.3001 | Train Acc: 10.30%


  -> Val Loss: 2.3048 | Val Acc: 9.90%


  -> Train Loss: 2.2999 | Train Acc: 10.30%


...

  -> Val Loss: 2.3050 | Val Acc: 9.90%


  -> Train Loss: 2.2986 | Train Acc: 10.30%


  -> Val Loss: 2.3050 | Val Acc: 10.00%
  [Save] Best val acc: 10.00%


  -> Train Loss: 2.2984 | Train Acc: 10.30%


  -> Val Loss: 2.3050 | Val Acc: 11.30%
  [Save] Best val acc: 11.30%
